# Notebook 01 — Residuals Are Not Noise

**Repo:** `residual-phase-lock`  
**Purpose:** demonstrate the first core claim:

> Residuals are not automatically noise. Residuals can reveal hidden structure.

This notebook builds a toy signal with a visible structural component, fits an intentionally incomplete baseline model, and shows that the residual preserves organized structure rather than behaving like random noise.

## Core idea

```text
model fit ≠ structure exhausted
residual ≠ noise
residual → hidden structure signal
```

## 1. Setup

We use only standard scientific Python packages so this notebook runs cleanly in Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)

## 2. Generate a signal with hidden structure

The signal has three pieces:

1. a simple linear trend,
2. a structured oscillatory component,
3. small random noise.

The baseline model will only fit the linear trend.  
The remaining residual should therefore contain structure.

In [ ]:
n = 500
x = np.linspace(0, 10, n)

trend = 0.8 * x + 1.5
hidden_structure = 0.9 * np.sin(2.5 * x) + 0.35 * np.sin(6.0 * x)
noise = np.random.normal(0, 0.25, size=n)

y = trend + hidden_structure + noise

X = x.reshape(-1, 1)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(x, y, label="observed signal", linewidth=1.5)
plt.plot(x, trend, label="true linear trend", linewidth=2)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Toy signal: trend + hidden structure + noise")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Fit an incomplete baseline model

We deliberately fit only a linear model.  
This gives the model a reasonable local fit while leaving hidden structure behind.

In [ ]:
model = LinearRegression()
model.fit(X, y)
y_hat = model.predict(X)

residual = y - y_hat

rmse = np.sqrt(mean_squared_error(y, y_hat))
r2 = r2_score(y, y_hat)

print(f"Baseline linear RMSE: {rmse:.4f}")
print(f"Baseline linear R²:   {r2:.4f}")
print(f"Fitted slope:         {model.coef_[0]:.4f}")
print(f"Fitted intercept:     {model.intercept_:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(x, y, label="observed signal", linewidth=1.5)
plt.plot(x, y_hat, label="linear fit", linewidth=2)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Incomplete baseline fit")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Inspect the residual

If the residual were only noise, it should look unstructured.  
Instead, it preserves coherent oscillatory organization.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(x, residual, label="residual", linewidth=1.5)
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("x")
plt.ylabel("residual")
plt.title("Residual after incomplete fit")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Compare residual to true hidden structure

In real data, we do not usually know hidden structure exactly.  
Here we do, because this is a controlled demonstration.

The comparison shows why residuals should be measured before being dismissed as noise.

In [ ]:
corr = np.corrcoef(residual, hidden_structure)[0, 1]

plt.figure(figsize=(10, 4))
plt.plot(x, hidden_structure, label="true hidden structure", linewidth=2)
plt.plot(x, residual, label="model residual", linewidth=1.5, alpha=0.85)
plt.xlabel("x")
plt.ylabel("value")
plt.title(f"Residual tracks hidden structure (correlation = {corr:.3f})")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Correlation(residual, hidden structure): {corr:.4f}")

## 6. Frequency-domain check

A random residual should not contain sharp organized peaks.  
Here, the residual has clear dominant frequencies from the hidden structure.

In [ ]:
residual_centered = residual - residual.mean()
freqs = np.fft.rfftfreq(n, d=(x[1] - x[0]))
spectrum = np.abs(np.fft.rfft(residual_centered))

plt.figure(figsize=(10, 4))
plt.plot(freqs, spectrum, linewidth=1.5)
plt.xlabel("frequency")
plt.ylabel("amplitude")
plt.title("Residual spectrum: structured peaks remain")
plt.xlim(0, 2)
plt.tight_layout()
plt.show()

top_indices = np.argsort(spectrum)[-5:][::-1]
dominant = pd.DataFrame({
    "frequency": freqs[top_indices],
    "amplitude": spectrum[top_indices]
})

dominant

## 7. Simple residual-structure score

We define a lightweight score:

```text
residual_structure_score = dominant spectral energy / total spectral energy
```

Larger values indicate that the residual energy is concentrated in organized modes rather than spread randomly.

In [ ]:
total_energy = np.sum(spectrum**2)
dominant_energy = np.sum(spectrum[top_indices[:3]]**2)
residual_structure_score = dominant_energy / total_energy

print(f"Residual structure score: {residual_structure_score:.4f}")

## 8. Phase-lock framing

In later notebooks, residuals become inputs to a phase-lock loop.

Notebook 01 establishes the first step:

```text
residual → structure signal
```

Notebook 02 should show a topology/global-structure failure.

Notebook 03 should add the revise loop:

```text
residual → detect mismatch
phase-lock → enforce alignment
revise → improve stability
```

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "baseline_rmse",
        "baseline_r2",
        "residual_hidden_structure_correlation",
        "residual_structure_score"
    ],
    "value": [
        rmse,
        r2,
        corr,
        residual_structure_score
    ]
})

summary

## 9. Takeaway

This notebook supports the repo's opening claim:

```text
residuals are not noise by default
residuals can reveal hidden structure
phase-lock begins by measuring that structure
```

The next notebook can move from residual structure to **topology failure** in a simple learning system.